# Interactive Tariff Visualizations
Interactive dashboard for analyzing Dutch energy tariffs by contract duration and supplier

## 1. Import Required Libraries

In [98]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore the JSON Data

In [99]:
# Load the JSON file
json_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2026\contracts_januari 2026.json'

with open(json_path, 'r', encoding='utf-8') as f:
    contracts_data = json.load(f)

print(f"Total contracts: {len(contracts_data)}")
print(f"\nFirst contract key: {list(contracts_data.keys())[0]}")
print(f"\nSample contract:")
sample = list(contracts_data.values())[0]
for key, value in sample.items():
    print(f"  {key}: {value}")

Total contracts: 207

First contract key: AllureNRG | Vaste Prijs 3 jaar | Vast (3 jaar)

Sample contract:
  Contractnaam: Vaste Prijs 3 jaar
  Contractduur: Vast (3 jaar)
  Geschatte leveringskosten obv verbruik: 152
  Vastrecht gas per jaar: 108,9000
  Variabel gas per m3: 1,2211
  Vastrecht elektriciteit per jaar: 108,9000
  Variabel elektriciteit enkel / piek per kWh: 0,2603
  Variabel elektriciteit dal per kWh: NaN
  _tupleId: 1
  _month_idx: januari 2026
  _session_id: 010D75AA71B74A289BABA5645521008F-0:0


## 3. Prepare Data for Visualization

In [100]:
# Convert JSON to DataFrame
df_list = []

for key, value in contracts_data.items():
    # Extract supplier name from the key format: "Supplier | ContractName | Duration"
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Geschatte leveringskosten obv verbruik': value.get('Geschatte leveringskosten obv verbruik', 'NaN'),
        'Vastrecht gas per jaar': value.get('Vastrecht gas per jaar', 'NaN'),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Vastrecht elektriciteit per jaar': value.get('Vastrecht elektriciteit per jaar', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
        'Variabel elektriciteit dal per kWh': value.get('Variabel elektriciteit dal per kWh', 'NaN'),
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

# Convert numeric columns, replacing 'NaN' with np.nan
numeric_columns = ['Geschatte leveringskosten obv verbruik', 'Vastrecht gas per jaar', 
                   'Variabel gas per m3', 'Vastrecht elektriciteit per jaar', 
                   'Variabel elektriciteit enkel / piek per kWh', 'Variabel elektriciteit dal per kWh']

for col in numeric_columns:
    df[col] = df[col].replace('NaN', np.nan)
    # Convert European format (comma decimal) to numeric
    df[col] = df[col].str.replace(',', '.') if df[col].dtype == 'object' else df[col]
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"DataFrame shape: {df.shape}")
print(f"\nUnique suppliers: {df['Leverancier'].nunique()}")
print(f"Unique contract durations: {df['Contractduur'].nunique()}")
print(f"\nContract durations: {sorted(df['Contractduur'].unique())}")
print(f"\nDataFrame info:")
df.info()

DataFrame shape: (207, 9)

Unique suppliers: 43
Unique contract durations: 5

Contract durations: ['Variabel (onbepaald)', 'Vast (1 jaar)', 'Vast (2 jaar)', 'Vast (3 jaar)', 'Vast (<1 jaar)']

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 9 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Leverancier                                  207 non-null    object 
 1   Contractnaam                                 207 non-null    object 
 2   Contractduur                                 207 non-null    object 
 3   Geschatte leveringskosten obv verbruik       207 non-null    int64  
 4   Vastrecht gas per jaar                       207 non-null    float64
 5   Variabel gas per m3                          207 non-null    float64
 6   Vastrecht elektriciteit per jaar             207 non-null    float64
 7   Variabel elektri

## 4. Create Interactive Category Selector

In [101]:
# Create dropdown widgets for filters
contractduur_options = ['All'] + sorted(df['Contractduur'].dropna().unique().tolist())
leverancier_options = ['All'] + sorted(df['Leverancier'].unique().tolist())

dropdown_contractduur = widgets.Dropdown(
    options=contractduur_options,
    value='All',
    description='Contract Duration:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

dropdown_leverancier = widgets.Dropdown(
    options=leverancier_options,
    value='All',
    description='Supplier:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

print("Filters ready! Use these dropdowns to select categories.")

Filters ready! Use these dropdowns to select categories.


## 5. Generate Bar Charts by Contract Duration

In [102]:
def plot_tariffs_by_duration(contractduur_filter):
    """Create bar charts for all contracts filtered by contract duration"""
    
    # Filter data
    if contractduur_filter != 'All':
        filtered_df = df[df['Contractduur'] == contractduur_filter].copy()
        title_suffix = f" - {contractduur_filter}"
    else:
        filtered_df = df.copy()
        title_suffix = " - All Contract Types"
    
    if len(filtered_df) == 0:
        return go.Figure().add_annotation(text="No data available for this selection")
    
    # Create label combining supplier and contract name for x-axis
    filtered_df['Label'] = filtered_df['Leverancier'] + ' | ' + filtered_df['Contractnaam']
    # Sort by gas tariff for better visualization
    filtered_df = filtered_df.sort_values('Variabel gas per m3', ascending=False).reset_index(drop=True)
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Gas Tariff (€/m³)", "Electricity Tariff (€/kWh)"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Gas tariff
    fig.add_trace(
        go.Bar(x=filtered_df['Label'], y=filtered_df['Variabel gas per m3'],
               marker_color='#FF6B6B', name='Gas'),
        row=1, col=1
    )
    
    # Electricity tariff
    fig.add_trace(
        go.Bar(x=filtered_df['Label'], y=filtered_df['Variabel elektriciteit enkel / piek per kWh'],
               marker_color='#4ECDC4', name='Electricity'),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="Supplier | Contract Name", row=1, col=1)
    fig.update_xaxes(title_text="Supplier | Contract Name", row=1, col=2)
    fig.update_yaxes(title_text="€/m³", row=1, col=1)
    fig.update_yaxes(title_text="€/kWh", row=1, col=2)
    
    fig.update_layout(
        title_text=f"All Contracts by Supplier{title_suffix}",
        height=600,
        showlegend=False,
        hovermode='x unified'
    )
    fig.update_xaxes(tickangle=-45)
    
    return fig

# Test with "All"
plot_tariffs_by_duration('All').show()

## 6. Generate Bar Charts by Supplier

In [103]:
def plot_tariffs_by_supplier(leverancier_filter):
    """Create bar charts for all contracts filtered by supplier"""
    
    # Filter data
    if leverancier_filter != 'All':
        filtered_df = df[df['Leverancier'] == leverancier_filter].copy()
        title_suffix = f" - {leverancier_filter}"
    else:
        filtered_df = df.copy()
        title_suffix = " - All Suppliers"
    
    if len(filtered_df) == 0:
        return go.Figure().add_annotation(text="No data available for this selection")
    
    # Sort by gas tariff for better visualization
    filtered_df = filtered_df.sort_values('Variabel gas per m3', ascending=False).reset_index(drop=True)
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Gas Tariff (€/m³)", "Electricity Tariff (€/kWh)"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Create custom labels for better readability
    labels = filtered_df['Leverancier'] + ' | ' + filtered_df['Contractnaam']
    
    # Gas tariff
    fig.add_trace(
        go.Bar(x=labels, y=filtered_df['Variabel gas per m3'],
               marker_color='#FF6B6B', name='Gas'),
        row=1, col=1
    )
    
    # Electricity tariff
    fig.add_trace(
        go.Bar(x=labels, y=filtered_df['Variabel elektriciteit enkel / piek per kWh'],
               marker_color='#4ECDC4', name='Electricity'),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="Supplier | Contract Name", row=1, col=1)
    fig.update_xaxes(title_text="Supplier | Contract Name", row=1, col=2)
    fig.update_yaxes(title_text="€/m³", row=1, col=1)
    fig.update_yaxes(title_text="€/kWh", row=1, col=2)
    
    fig.update_layout(
        title_text=f"All Contracts{title_suffix}",
        height=600,
        showlegend=False,
        hovermode='x unified'
    )
    fig.update_xaxes(tickangle=-45)
    
    return fig

# Test with "All"
plot_tariffs_by_supplier('All').show()

## 7. Combine Visualizations with Filters

In [104]:
### Interactive Dashboard - Filter by Contract Duration
display(widgets.VBox([
    widgets.HTML("<h3>All Contracts (filtered by Contract Duration)</h3>"),
    dropdown_contractduur
]))

output_duration = widgets.Output()
display(output_duration)

def on_contractduur_change(change):
    with output_duration:
        output_duration.clear_output(wait=True)
        plot_tariffs_by_duration(change['new']).show()

dropdown_contractduur.observe(on_contractduur_change, names='value')
# Initial plot
with output_duration:
    plot_tariffs_by_duration('All').show()

print("\nChange the dropdown above to filter by contract duration!")

Output()


Change the dropdown above to filter by contract duration!


In [105]:
### Interactive Dashboard - Filter by Supplier
display(widgets.VBox([
    widgets.HTML("<h3>All Contracts (filtered by Supplier)</h3>"),
    dropdown_leverancier
]))

output_supplier = widgets.Output()
display(output_supplier)

def on_leverancier_change(change):
    with output_supplier:
        output_supplier.clear_output(wait=True)
        plot_tariffs_by_supplier(change['new']).show()

dropdown_leverancier.observe(on_leverancier_change, names='value')
# Initial plot
with output_supplier:
    plot_tariffs_by_supplier('All').show()

print("\nChange the dropdown above to filter by supplier!")

Output()


Change the dropdown above to filter by supplier!


## 8. Summary Statistics

In [106]:
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"Total contracts: {len(df)}")
print(f"Total suppliers: {df['Leverancier'].nunique()}")
print(f"Total contract types: {df['Contractduur'].nunique()}")

print("\n" + "=" * 80)
print("GAS TARIFFS (€/m³)")
gas_col = 'Variabel gas per m3'
print(f"  Average: €{df[gas_col].mean():.4f}")
print(f"  Min: €{df[gas_col].min():.4f}")
print(f"  Max: €{df[gas_col].max():.4f}")
print(f"  Median: €{df[gas_col].median():.4f}")

print("\n" + "=" * 80)
print("ELECTRICITY TARIFFS (€/kWh)")
elec_col = 'Variabel elektriciteit enkel / piek per kWh'
print(f"  Average: €{df[elec_col].mean():.4f}")
print(f"  Min: €{df[elec_col].min():.4f}")
print(f"  Max: €{df[elec_col].max():.4f}")
print(f"  Median: €{df[elec_col].median():.4f}")

print("\n" + "=" * 80)
print("CONTRACTS BY TYPE")
for duration in sorted(df['Contractduur'].dropna().unique()):
    count = len(df[df['Contractduur'] == duration])
    print(f"  {duration}: {count} contracts")

print("\n" + "=" * 80)
print("CONTRACTS BY SUPPLIER (Top 10)")
supplier_counts = df['Leverancier'].value_counts().head(10)
for supplier, count in supplier_counts.items():
    print(f"  {supplier}: {count} contracts")

SUMMARY STATISTICS
Total contracts: 207
Total suppliers: 43
Total contract types: 5

GAS TARIFFS (€/m³)
  Average: €1.2792
  Min: €1.0752
  Max: €1.8158
  Median: €1.2717

ELECTRICITY TARIFFS (€/kWh)
  Average: €0.2710
  Min: €0.1230
  Max: €0.5586
  Median: €0.2657

CONTRACTS BY TYPE
  Variabel (onbepaald): 87 contracts
  Vast (1 jaar): 64 contracts
  Vast (2 jaar): 15 contracts
  Vast (3 jaar): 39 contracts
  Vast (<1 jaar): 2 contracts

CONTRACTS BY SUPPLIER (Top 10)
  Greenchoice: 21 contracts
  Greenchoice Zakelijk: 14 contracts
  Energiedirect.nl: 13 contracts
  Essent: 12 contracts
  Mega: 11 contracts
  UnitedConsumers: 10 contracts
  Eneco: 9 contracts
  Hezelaer Energy: 8 contracts
  Noord Energie: 7 contracts
  Budget Energie: 7 contracts


## 9. Load Historical Data - 3 Month Timeline

In [107]:
# Load September 2025, Oktober 2025, November 2025, December 2025, and January 2026 data
september_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_september 2025.json'
oktober_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_oktober 2025.json'
november_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_november 2025.json'
december_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_december 2025.json'
january_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2026\contracts_januari 2026.json'

# Load files
with open(september_path, 'r', encoding='utf-8') as f:
    september_data = json.load(f)

with open(oktober_path, 'r', encoding='utf-8') as f:
    oktober_data = json.load(f)

with open(november_path, 'r', encoding='utf-8') as f:
    november_data = json.load(f)

with open(december_path, 'r', encoding='utf-8') as f:
    december_data = json.load(f)

with open(january_path, 'r', encoding='utf-8') as f:
    january_data = json.load(f)

# Create timeline dataframe
timeline_df_list = []

# Process September 2025
for key, value in september_data.items():
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Month': 'September 2025',
        'Month_Num': 0,
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
    }
    timeline_df_list.append(row)

# Process Oktober 2025
for key, value in oktober_data.items():
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Month': 'Oktober 2025',
        'Month_Num': 1,
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
    }
    timeline_df_list.append(row)

# Process November 2025
for key, value in november_data.items():
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Month': 'November 2025',
        'Month_Num': 2,
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
    }
    timeline_df_list.append(row)

# Process December 2025
for key, value in december_data.items():
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Month': 'December 2025',
        'Month_Num': 3,
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
    }
    timeline_df_list.append(row)

# Process January 2026
for key, value in january_data.items():
    parts = key.split(' | ')
    leverancier = parts[0] if len(parts) > 0 else 'Unknown'
    
    row = {
        'Month': 'January 2026',
        'Month_Num': 4,
        'Leverancier': leverancier,
        'Contractnaam': value.get('Contractnaam', ''),
        'Contractduur': value.get('Contractduur', ''),
        'Variabel gas per m3': value.get('Variabel gas per m3', 'NaN'),
        'Variabel elektriciteit enkel / piek per kWh': value.get('Variabel elektriciteit enkel / piek per kWh', 'NaN'),
    }
    timeline_df_list.append(row)

df_timeline = pd.DataFrame(timeline_df_list)

# Convert numeric columns
numeric_cols = ['Variabel gas per m3', 'Variabel elektriciteit enkel / piek per kWh']
for col in numeric_cols:
    df_timeline[col] = df_timeline[col].replace('NaN', np.nan)
    df_timeline[col] = df_timeline[col].astype(str).str.replace(',', '.')
    df_timeline[col] = pd.to_numeric(df_timeline[col], errors='coerce')

print(f"Timeline data loaded:")
print(f"  September 2025: {len(september_data)} contracts")
print(f"  Oktober 2025: {len(oktober_data)} contracts")
print(f"  November 2025: {len(november_data)} contracts")
print(f"  December 2025: {len(december_data)} contracts")
print(f"  January 2026: {len(january_data)} contracts")
print(f"  Total: {len(df_timeline)} records")
print(f"  Unique suppliers: {df_timeline['Leverancier'].nunique()}")

Timeline data loaded:
  September 2025: 207 contracts
  Oktober 2025: 212 contracts
  November 2025: 184 contracts
  December 2025: 195 contracts
  January 2026: 207 contracts
  Total: 1005 records
  Unique suppliers: 47


In [108]:
## 10. Create Timeline Filters

In [109]:
# Create dropdown for contract duration
timeline_duration_options = ['All'] + sorted(df_timeline['Contractduur'].dropna().unique().tolist())
timeline_supplier_options = ['All'] + sorted(df_timeline['Leverancier'].unique().tolist())

dropdown_timeline_duration = widgets.Dropdown(
    options=timeline_duration_options,
    value='All',
    description='Contract Duration:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

dropdown_timeline_supplier = widgets.Dropdown(
    options=timeline_supplier_options,
    value='All',
    description='Supplier:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Container for contract name checkboxes (will be populated dynamically)
contract_names_container = widgets.VBox([widgets.HTML("<b>Contract Names:</b>")])

def update_contract_names(supplier, duration):
    """Update available contract names based on selected supplier AND duration"""
    filtered = df_timeline.copy()
    
    if supplier != 'All':
        filtered = filtered[filtered['Leverancier'] == supplier]
    
    if duration != 'All':
        filtered = filtered[filtered['Contractduur'] == duration]
    
    names = sorted(filtered['Contractnaam'].dropna().unique().tolist())
    
    if len(names) == 0:
        return [], []
    
    # Create checkboxes for each contract name
    checkboxes = [
        widgets.Checkbox(value=True, description=name, layout=widgets.Layout(width='90%'))
        for name in names
    ]
    return names, checkboxes

print("Timeline filters created!")

Timeline filters created!


## 11. Create Timeline Plotting Function

In [110]:
def plot_timeline(duration_filter, supplier_filter, contract_names_filter):
    """Create timeline plot of tariffs over September 2025 - January 2026"""
    
    # Start with full dataset
    filtered_df = df_timeline.copy()
    
    # Apply filters
    if duration_filter != 'All':
        filtered_df = filtered_df[filtered_df['Contractduur'] == duration_filter]
    
    if supplier_filter != 'All':
        filtered_df = filtered_df[filtered_df['Leverancier'] == supplier_filter]
    
    if contract_names_filter and len(contract_names_filter) > 0:
        filtered_df = filtered_df[filtered_df['Contractnaam'].isin(contract_names_filter)]
    
    if len(filtered_df) == 0:
        return go.Figure().add_annotation(text="No data available for this selection")
    
    # Create subplots for gas and electricity
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Gas Tariff (€/m³)", "Electricity Tariff (€/kWh)"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Define chronological month order
    month_order = ['September 2025', 'Oktober 2025', 'November 2025', 'December 2025', 'January 2026']
    
    # Group by supplier and contract name for line traces
    for (leverancier, contractnaam), group in filtered_df.groupby(['Leverancier', 'Contractnaam']):
        group = group.sort_values('Month_Num')
        label = f"{leverancier} | {contractnaam}"
        
        # Gas tariff line
        fig.add_trace(
            go.Scatter(
                x=group['Month'],
                y=group['Variabel gas per m3'],
                mode='lines+markers',
                name=label,
                hovertemplate='<b>' + label + '</b><br>%{x}<br>€%{y:.4f}/m³<extra></extra>',
                line=dict(width=2)
            ),
            row=1, col=1
        )
        
        # Electricity tariff line
        fig.add_trace(
            go.Scatter(
                x=group['Month'],
                y=group['Variabel elektriciteit enkel / piek per kWh'],
                mode='lines+markers',
                name=label,
                hovertemplate='<b>' + label + '</b><br>%{x}<br>€%{y:.4f}/kWh<extra></extra>',
                line=dict(width=2),
                showlegend=False
            ),
            row=1, col=2
        )
    
    fig.update_xaxes(title_text="Month", row=1, col=1)
    fig.update_xaxes(title_text="Month", row=1, col=2)
    fig.update_yaxes(title_text="€/m³", row=1, col=1)
    fig.update_yaxes(title_text="€/kWh", row=1, col=2)
    
    # Set chronological month order on x-axis
    fig.update_xaxes(categoryorder="array", categoryarray=month_order, row=1, col=1)
    fig.update_xaxes(categoryorder="array", categoryarray=month_order, row=1, col=2)
    
    title = "Tariff Timeline: September 2025 - January 2026"
    if supplier_filter != 'All':
        title += f" ({supplier_filter})"
    if duration_filter != 'All':
        title += f" - {duration_filter}"
    
    fig.update_layout(
        title_text=title,
        height=600,
        hovermode='x unified',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return fig

# Test plot
plot_timeline('All', 'All', None).show()

## 12. Interactive Timeline Dashboard (November - January)

In [111]:
# Initialize contract name checkboxes with all contracts
initial_names, initial_checkboxes = update_contract_names('All', 'All')
checkboxes_vbox = widgets.VBox(initial_checkboxes)

# Output areas for dynamic updates
contract_names_output = widgets.VBox([widgets.HTML("<b>Contract Names:</b>"), checkboxes_vbox])
plot_output = widgets.Output()

# Display header and filters
display(widgets.HTML("<h2>Tariff Timeline: September 2025 - January 2026</h2>"))
display(widgets.VBox([
    dropdown_timeline_duration,
    dropdown_timeline_supplier,
]))
display(contract_names_output)
display(plot_output)

def refresh_contract_names():
    """Refresh contract names based on current supplier and duration selections"""
    supplier = dropdown_timeline_supplier.value
    duration = dropdown_timeline_duration.value
    
    names, checkboxes = update_contract_names(supplier, duration)
    contract_names_output.children = (
        widgets.HTML("<b>Contract Names:</b>"),
        widgets.VBox(checkboxes) if checkboxes else widgets.HTML("<i>No contracts available for this combination</i>")
    )
    update_timeline_plot()

def on_supplier_change(change):
    """Update contract name checkboxes when supplier changes"""
    refresh_contract_names()

def on_duration_change(change):
    """Update contract name checkboxes when duration changes"""
    refresh_contract_names()

def update_timeline_plot(*args):
    """Update timeline plot based on current filter values"""
    with plot_output:
        plot_output.clear_output(wait=True)
        
        duration = dropdown_timeline_duration.value
        supplier = dropdown_timeline_supplier.value
        
        # Get selected contract names from checkboxes
        contract_names = []
        for child in contract_names_output.children[1].children if len(contract_names_output.children) > 1 else []:
            if isinstance(child, widgets.Checkbox) and child.value:
                contract_names.append(child.description)
        
        plot_timeline(duration, supplier, contract_names).show()

# Register event handlers for both filters
dropdown_timeline_duration.observe(on_duration_change, names='value')
dropdown_timeline_supplier.observe(on_supplier_change, names='value')

# Initial plot
with plot_output:
    plot_timeline('All', 'All', initial_names).show()

print("\nTimeline dashboard ready! Adjust filters to update the plot.")

HTML(value='<h2>Tariff Timeline: September 2025 - January 2026</h2>')

Output()


Timeline dashboard ready! Adjust filters to update the plot.
